## 📋 IMPORTANT: Execution Order

**Run cells in this order** (scroll down to find them):
1. ✅ **Import Required Libraries** (bottom of notebook)
2. ✅ **Helper Functions for Preprocessing**
3. ✅ **Main Processing Function - Process All 600 Cases**
4. ✅ **Run Preprocessing on All 600 Cases** (this will take ~15-30 minutes)
5. ✅ **Visualize Sample Results**
6. ✅ **Generate Summary Statistics** (this cell at top)

💡 **Tip:** Use "Run All" from the menu to execute all cells in order automatically!

## 1. Import Required Libraries

In [ ]:
import os
import json
import numpy as np
import nibabel as nib
from pathlib import Path
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt

## 2. Helper Functions for Preprocessing

In [ ]:
def hu_to_grayscale(hu_values, window_min=-135, window_max=215):
    """Convert HU values to grayscale [0, 255] using intensity windowing."""
    windowed = np.clip(hu_values, window_min, window_max)
    normalized = (windowed - window_min) / (window_max - window_min)
    grayscale = (normalized * 255).astype(np.uint8)
    return grayscale


def find_best_axial_slice(organ_mask):
    """Find the axial slice index with maximum organ area.
    
    For AMOS NIFTI files with orientation (L, A, S):
    - Axis 2: Superior-Inferior (z) - this is the axial direction
    
    Returns the z-index where the organ has the largest cross-sectional area.
    """
    max_area = 0
    best_z = 0
    
    for z in range(organ_mask.shape[2]):
        area = np.sum(organ_mask[:, :, z] > 0)
        if area > max_area:
            max_area = area
            best_z = z
    
    return best_z


def get_square_bbox_2d(mask_slice):
    """Create a square bounding box around the organ in a 2D slice.
    
    Args:
        mask_slice: 2D binary mask (H x W)
    
    Returns:
        Tuple of (x_min, x_max, y_min, y_max) for square bounding box
    """
    # Find bounding box coordinates
    x_coords = np.any(mask_slice, axis=1)
    y_coords = np.any(mask_slice, axis=0)
    
    x_idx = np.where(x_coords)[0]
    y_idx = np.where(y_coords)[0]
    
    if len(x_idx) == 0 or len(y_idx) == 0:
        return None
    
    x_min, x_max = x_idx[0], x_idx[-1]
    y_min, y_max = y_idx[0], y_idx[-1]
    
    # Calculate dimensions
    x_size = x_max - x_min + 1
    y_size = y_max - y_min + 1
    
    # Make it square - use the maximum dimension
    max_size = max(x_size, y_size)
    
    # Center the square
    x_center = (x_min + x_max) // 2
    y_center = (y_min + y_max) // 2
    
    # Calculate new bounds for square box
    half_size = max_size // 2
    
    x_min_square = max(0, x_center - half_size)
    x_max_square = min(mask_slice.shape[0] - 1, x_center + half_size)
    
    y_min_square = max(0, y_center - half_size)
    y_max_square = min(mask_slice.shape[1] - 1, y_center + half_size)
    
    return (x_min_square, x_max_square, y_min_square, y_max_square)


def extract_organ_slice_with_square_bbox(image_data, organ_mask):
    """Extract the best axial slice and crop to square bounding box.
    
    Process:
    1. Find axial slice with maximum organ area
    2. Create square bounding box around organ in that slice
    3. Crop both image and mask to the square box
    
    Returns:
        Tuple of (image_crop, mask_crop)
    """
    # Step 1: Find best axial slice
    best_z = find_best_axial_slice(organ_mask)
    
    # Extract 2D slices at best z-level
    image_slice = image_data[:, :, best_z]
    mask_slice = organ_mask[:, :, best_z]
    
    # Step 2: Get square bounding box in 2D
    bbox = get_square_bbox_2d(mask_slice)
    
    if bbox is None:
        return None, None
    
    x_min, x_max, y_min, y_max = bbox
    
    # Step 3: Crop to square bounding box
    image_crop = image_slice[x_min:x_max+1, y_min:y_max+1]
    mask_crop = mask_slice[x_min:x_max+1, y_min:y_max+1]
    
    return image_crop, mask_crop


def resize_image(image, target_size=(224, 224)):
    """Resize image to target size using bilinear interpolation."""
    pil_image = Image.fromarray(image)
    resized = pil_image.resize((target_size[1], target_size[0]), Image.BILINEAR)
    return np.array(resized)


def get_organ_label(segmentation_slice):
    """Get multi-hot label vector for organs present in the slice."""
    label_vector = np.zeros(15, dtype=np.uint8)
    unique_labels = np.unique(segmentation_slice)
    unique_labels = unique_labels[unique_labels > 0]
    
    for label in unique_labels:
        if 1 <= label <= 15:
            label_vector[label - 1] = 1
    
    return label_vector

## 3. Main Processing Function

In [ ]:
def process_amos_dataset(amos_root, output_path):
    """Process ONLY CT scans (ID < 500) from AMOS dataset and save in OrganaMNIST-compatible format.
    
    For each patient, creates ONE IMAGE PER ORGAN:
    - Find the axial slice with maximum organ area
    - Create a square bounding box around the organ in 2D
    - Resize to 224x224 with bilinear interpolation
    - Save as a separate 224x224 image with single-organ label
    """
    
    # Load dataset.json
    dataset_json_path = os.path.join(amos_root, 'dataset.json')
    with open(dataset_json_path, 'r') as f:
        dataset_info = json.load(f)
    
    # Collect samples from training and validation sets - ONLY CT (ID < 500)
    all_samples = []
    
    # Training set
    for item in dataset_info['training']:
        image_path = os.path.join(amos_root, item['image'].replace('./', ''))
        label_path = os.path.join(amos_root, item['label'].replace('./', ''))
        sample_id = int(os.path.basename(image_path).split('_')[1].split('.')[0])
        if sample_id < 500:  # Only CT scans
            all_samples.append({'image': image_path, 'label': label_path})
    
    # Validation set
    for item in dataset_info['validation']:
        image_path = os.path.join(amos_root, item['image'].replace('./', ''))
        label_path = os.path.join(amos_root, item['label'].replace('./', ''))
        sample_id = int(os.path.basename(image_path).split('_')[1].split('.')[0])
        if sample_id < 500:  # Only CT scans
            all_samples.append({'image': image_path, 'label': label_path})
    
    print(f"🔍 Total CT samples to process: {len(all_samples)} (ID < 500)")
    print(f"📊 Each sample will generate multiple images (one per organ)")
    print(f"⚠️  Note: MRI scans (ID ≥ 500) and unlabeled Test set excluded\n")
    
    # Organ names for reference
    organ_names = ['spleen', 'right kidney', 'left kidney', 'gall bladder', 'esophagus', 
                   'liver', 'stomach', 'aorta', 'postcava', 'pancreas', 'right adrenal gland', 
                   'left adrenal gland', 'duodenum', 'bladder', 'prostate/uterus']
    
    # Process each sample - generate one image per organ
    processed_images = []
    processed_labels = []
    sample_ids = []
    failed_samples = []
    skipped_organs = 0
    
    for sample in tqdm(all_samples, desc="Processing CT samples"):
        try:
            # Load image and segmentation
            image_nifti = nib.load(sample['image'])
            image_data = image_nifti.get_fdata()
            
            seg_nifti = nib.load(sample['label'])
            seg_data = seg_nifti.get_fdata().astype(np.uint8)
            
            # Extract patient ID
            filename = os.path.basename(sample['image'])
            patient_id = filename.split('_')[1].split('.')[0]
            
            # Find all unique organs in this patient
            unique_organs = np.unique(seg_data)
            unique_organs = unique_organs[(unique_organs > 0) & (unique_organs <= 15)]
            
            if len(unique_organs) == 0:
                print(f"\n⚠️  Warning: No organs found in {sample['image']}")
                failed_samples.append(sample['image'])
                continue
            
            # Process each organ separately
            for organ_id in unique_organs:
                try:
                    # Create binary mask for this specific organ
                    organ_mask = (seg_data == organ_id).astype(np.uint8)
                    
                    # Extract best slice and crop to square bounding box
                    image_crop, mask_crop = extract_organ_slice_with_square_bbox(image_data, organ_mask)
                    
                    if image_crop is None or mask_crop is None:
                        print(f"\n⚠️  Warning: Could not extract slice for organ {organ_id} in {patient_id}")
                        skipped_organs += 1
                        continue
                    
                    # Verify that the organ is actually present in the cropped slice
                    if not np.any(mask_crop):
                        skipped_organs += 1
                        continue
                    
                    # Convert HU to grayscale
                    grayscale_crop = hu_to_grayscale(image_crop, window_min=-135, window_max=215)
                    
                    # Resize to 224x224 - BOTH IMAGE AND SEGMENTATION
                    resized_image = resize_image(grayscale_crop, target_size=(224, 224))
                    resized_seg = resize_image(mask_crop, target_size=(224, 224))
                    
                    # Verify organ is still present after resizing
                    if not np.any(resized_seg > 0):
                        skipped_organs += 1
                        continue
                    
                    # Create single-organ label (one-hot encoding with only this organ)
                    organ_label = np.zeros(15, dtype=np.uint8)
                    organ_label[organ_id - 1] = 1
                    
                    # Store results
                    processed_images.append(resized_image)
                    processed_labels.append(organ_label)
                    sample_ids.append(f"{patient_id}_organ{organ_id:02d}")
                    
                except Exception as e:
                    print(f"\n❌ Error processing organ {organ_id} in {patient_id}: {e}")
                    skipped_organs += 1
                    continue
            
        except Exception as e:
            print(f"\n❌ Error processing {sample['image']}: {e}")
            failed_samples.append(sample['image'])
            continue
    
    print(f"\n✅ Successfully processed: {len(processed_images)} organ images from {len(all_samples)} patients")
    print(f"⚠️  Skipped {skipped_organs} organs")
    print(f"❌ Failed: {len(failed_samples)} patient scans")
    
    if len(failed_samples) > 0:
        print("\nFailed samples:")
        for fail in failed_samples[:10]:
            print(f"  - {fail}")
        if len(failed_samples) > 10:
            print(f"  ... and {len(failed_samples) - 10} more")
    
    # Convert to numpy arrays
    images_array = np.array(processed_images, dtype=np.uint8)
    labels_array = np.array(processed_labels, dtype=np.uint8)
    sample_ids_array = np.array(sample_ids)
    
    # Add channel dimension if grayscale (H, W) -> (H, W, 1)
    if images_array.ndim == 3:
        images_array = images_array[..., np.newaxis]
    
    print(f"\n📐 Final shapes:")
    print(f"  Images: {images_array.shape}")
    print(f"  Labels: {labels_array.shape}")
    print(f"  Average organs per patient: {len(processed_images) / len(all_samples):.1f}")
    
    # Save as NPZ file (following MedMNIST convention)
    np.savez_compressed(
        output_path,
        test_images=images_array,
        test_labels=labels_array,
        test_sample_ids=sample_ids_array
    )
    
    print(f"\n💾 Saved processed dataset to: {output_path}")
    
    # Save metadata JSON
    metadata = {
        'source': 'AMOS_2022',
        'description': 'AMOS CT-only dataset (ID < 500) - ONE IMAGE PER ORGAN per patient',
        'preprocessing': {
            'hu_window_min': -135,
            'hu_window_max': 215,
            'target_size': [224, 224],
            'interpolation': 'bilinear',
            'slice_selection': 'best_axial_slice_with_maximum_organ_area',
            'bounding_box': 'square_2D_bbox_after_slice_selection',
            'note': 'Process: 1) Find slice with max organ area, 2) Create square bbox in 2D, 3) Resize to 224x224'
        },
        'modality': 'CT',
        'id_filter': 'ID < 500 (CT scans only, MRI excluded)',
        'n_samples': len(processed_images),
        'n_patients': len(all_samples),
        'n_failed': len(failed_samples),
        'n_skipped_organs': skipped_organs,
        'splits_merged': ['training', 'validation'],
        'organ_labels': {
            '0': 'spleen',
            '1': 'right kidney',
            '2': 'left kidney',
            '3': 'gall bladder',
            '4': 'esophagus',
            '5': 'liver',
            '6': 'stomach',
            '7': 'aorta',
            '8': 'postcava',
            '9': 'pancreas',
            '10': 'right adrenal gland',
            '11': 'left adrenal gland',
            '12': 'duodenum',
            '13': 'bladder',
            '14': 'prostate/uterus'
        }
    }
    
    metadata_path = output_path.replace('.npz', '_metadata.json')
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print(f"📝 Saved metadata to: {metadata_path}")
    
    return images_array, labels_array, sample_ids_array

## 4. Run Preprocessing on All Cases

In [ ]:
# Set paths
amos_root = '/mnt/data/psteinmetz/computer_vision_code/code/UQ_Toolbox/medMNIST/AMOS_2022/amos22/amos22'
output_path = '/mnt/data/psteinmetz/computer_vision_code/code/UQ_Toolbox/medMNIST/AMOS_2022/amos_external_test_224.npz'

# Run preprocessing on ALL samples
images, labels, sample_ids = process_amos_dataset(amos_root, output_path)

## 5. Visualize Sample Results

In [ ]:
# Visualize some samples
n_samples = 15
fig, axes = plt.subplots(3, 5, figsize=(16, 12))
axes = axes.flatten()

organ_names = ['spleen', 'right kidney', 'left kidney', 'gall bladder', 'esophagus', 
               'liver', 'stomach', 'aorta', 'postcava', 'pancreas', 'right adrenal', 
               'left adrenal', 'duodenum', 'bladder', 'prostate/uterus']

for i in range(n_samples):
    axes[i].imshow(images[i, :, :, 0], cmap='gray')
    
    # Get organ labels
    present_organs = [organ_names[j] for j in range(15) if labels[i, j] == 1]
    title = f"ID: {sample_ids[i]}\n" + ", ".join(present_organs[:3])
    if len(present_organs) > 3:
        title += f"\n+{len(present_organs)-3} more"
    
    axes[i].set_title(title, fontsize=9)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(os.path.dirname(output_path), 'sample_preprocessed.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Generate Summary Statistics

In [ ]:
print("=" * 60)
print("SUMMARY STATISTICS FOR PROCESSED AMOS DATASET")
print("=" * 60)

print(f"\n📊 Dataset Overview:")
print(f"  Total processed samples: {len(images)}")
print(f"  Image shape: {images.shape}")
print(f"  Label shape: {labels.shape}")

print(f"\n🏷️  Organ Label Distribution:")
organs_per_image = labels.sum(axis=1)
print(f"  Organs per image - Min: {organs_per_image.min()}")
print(f"  Organs per image - Max: {organs_per_image.max()}")
print(f"  Organs per image - Mean: {organs_per_image.mean():.2f}")
print(f"  Organs per image - Median: {np.median(organs_per_image):.0f}")

print(f"\n🎯 Individual Organ Frequencies:")
organ_counts = labels.sum(axis=0)
for i, (name, count) in enumerate(zip(organ_names, organ_counts)):
    percentage = (count / len(labels)) * 100
    print(f"  {i+1:2d}. {name:20s}: {count:3d} ({percentage:5.1f}%)")

# Plot organ distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Histogram of organs per image
ax1.hist(organs_per_image, bins=range(organs_per_image.min(), organs_per_image.max()+2), 
         edgecolor='black', alpha=0.7)
ax1.set_xlabel('Number of Organs per Image', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Distribution of Organs per Image', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Bar plot of individual organ frequencies
ax2.barh(organ_names, organ_counts, edgecolor='black', alpha=0.7)
ax2.set_xlabel('Number of Images', fontsize=12)
ax2.set_title('Organ Frequency Across All Images', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(os.path.dirname(output_path), 'statistics_summary.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Processing complete! Dataset ready for use as external test set.")